# gemynd — newspaper ingest on an A100 with e4b (vLLM, batched)

Runs gemynd's claim-extraction pipeline over a large newspaper corpus using the
**e4b** model served by **vLLM** on a single **A100**. This is the scale-out path
for the local e4b config we validated: same model, but vLLM's continuous batching
turns dozens of paragraphs into concurrent requests, so throughput is 1–2 orders
of magnitude above a single consumer GPU.

**What this notebook does**
1. Serve e4b on vLLM (OpenAI-compatible, structured output, prefix caching).
2. Point gemynd at it with the tuned bulk-ingest config (`min_epi` schema profile).
3. Ingest the corpus from **Google Drive**, writing resumable `*.semantic.json`
   bundles back to Drive (memory backend — no Neo4j needed on Colab).

**Resume-safe:** the checkpoint and claim-cache live on Drive, so a preempted
runtime picks up exactly where it left off — just re-run the ingest cell.


## 0. Config — edit these, then Run-All

In [ ]:
# ============================ CONFIG ============================
# Repo: a copy of the gemynd repo reachable from Colab. Either git-clone it
# (set GIT_URL below) or keep a copy in Drive and point REPO at it.
REPO       = "/content/gemynd/Archive-Claim-Graph"
GIT_URL    = ""   # e.g. "https://github.com/<you>/gemynd.git"; leave "" to use a Drive copy

# Corpus + outputs live in Google Drive (mounted below).
CORPUS_DIR = "/content/drive/MyDrive/gemynd/corpus"                 # your .json / .jsonl bundles
OUT_DIR    = "/content/drive/MyDrive/gemynd/out/newspapers_e4b"     # bundles + checkpoint (resume-safe)

# Domain config for THIS corpus. resources_newspaper for newspapers; change if the
# Drive corpus is a different domain.
DOMAIN_DIR = f"{REPO}/gemynd/resources_newspaper"

# e4b served by vLLM. "gemma-4-e4b" in LM Studio == this HF repo (Gemma 3n E4B).
MODEL      = "google/gemma-3n-E4B-it"
PORT       = 8000

WORKERS        = 8         # parallel worker PROCESSES feeding vLLM's batch (~ Colab vCPU count)
SCHEMA_PROFILE = "min_epi" # tuned: ~40% fewer output tokens vs 'full', same claim yield
MAX_MODEL_LEN  = 4096
MAX_NUM_SEQS   = 64        # how many requests vLLM batches concurrently (raise to fill the A100)
print("config loaded")


## 1. GPU check — confirm you actually have an A100

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install vLLM + get the repo

Install a **current** vLLM (don't pin an old version — it breaks against Colab's
newer `transformers`). Gemma-3n needs a recent vLLM (≥ 0.8.x). The red
`dependency conflict` wall Colab prints (numpy / jax / opencv / …) is expected
and harmless — this notebook only *serves* a model and never imports those.


In [ ]:
!pip install -q -U vllm
import os, subprocess
# Get the repo: git clone if a URL is given, else expect a Drive-mounted copy.
if GIT_URL:
    os.makedirs("/content/gemynd", exist_ok=True)
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "--depth", "1", GIT_URL, "/content/gemynd"], check=True)
print("REPO exists:", os.path.isdir(REPO))


## 3. Mount Google Drive (corpus in, bundles out) + install gemynd

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.isdir(REPO), f"REPO not found: {REPO} — clone it or copy it into Drive."
assert os.path.isdir(CORPUS_DIR), f"CORPUS_DIR not found: {CORPUS_DIR}"
# Install gemynd's runtime deps (editable install so `python -m gemynd.cli` works).
!cd {REPO} && pip install -q -e . || pip install -q -r {REPO}/requirements.txt
print("drive mounted; repo + corpus present")


## 4. Serve e4b on vLLM

`--enable-prefix-caching` caches the shared ~640-token system prompt across every
call (near-free after the first). `--max-num-seqs` sets how many requests vLLM
batches at once — the lever that fills the A100.

> **If gemma-3n won't serve here** (arch/version issues): fall back to a llama.cpp
> server with `--parallel {MAX_NUM_SEQS}` on the e4b GGUF (still batches, slightly
> less efficient), or swap `MODEL` to a vLLM-native small instruct model. The
> smoke tests below fail fast so you find out before the full run.


In [ ]:
import subprocess, time, urllib.request
server = subprocess.Popen([
    "vllm", "serve", MODEL,
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", "0.90",
    "--enable-prefix-caching",
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--port", str(PORT),
])
base = f"http://localhost:{PORT}/v1"
for _ in range(180):  # model download + load can take several minutes
    try:
        urllib.request.urlopen(f"{base}/models", timeout=5); print("\nvLLM up at", base); break
    except Exception:
        time.sleep(10)
else:
    raise RuntimeError("vLLM did not start — check output / GPU memory / MODEL id.")


## 5. Smoke tests — endpoint + **structured output** (the risky bit)

In [ ]:
import json, urllib.request
def post(payload):
    req = urllib.request.Request(f"{base}/chat/completions",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json", "Authorization": "Bearer not-needed"})
    return json.load(urllib.request.urlopen(req, timeout=120))

# 5a. plain chat
print("chat:", post({"model": MODEL, "messages": [{"role": "user", "content": "Reply with one word: ready"}],
                     "max_tokens": 8, "temperature": 0})["choices"][0]["message"]["content"])

# 5b. array-root json_schema — EXACTLY the shape gemynd's adapter sends. If this
#     404/400s, vLLM's structured-output backend rejects array roots; see note in cell 4.
schema = {"type": "json_schema", "json_schema": {"name": "claim_extraction", "strict": True,
    "schema": {"type": "array", "items": {"type": "object", "additionalProperties": False,
    "required": ["source_sentence"], "properties": {
        "source_sentence": {"type": "string"},
        "claim_type": {"type": "string"},
        "epistemic_status": {"type": "string", "enum": ["observed", "estimated", "uncertain", "reported"]},
        "claim_links": {"type": "array", "items": {"type": "object", "additionalProperties": False,
            "required": ["surface_form"], "properties": {"surface_form": {"type": "string"}}}}}}}}}
out = post({"model": MODEL, "messages": [
        {"role": "system", "content": "Extract factual claims as a JSON array of the given shape."},
        {"role": "user", "content": "On March 3, 1889 President Cleveland signed the act admitting Washington as a state."}],
    "max_tokens": 512, "temperature": 0, "response_format": schema})
raw = out["choices"][0]["message"]["content"]
print("structured-output parsed:", json.loads(raw))  # raises if not a valid array -> stop and fix


## 6. Point gemynd at the vLLM endpoint (tuned bulk-ingest config)

In [ ]:
import os
os.environ["CLAIM_LLM_BASE_URL"]      = base            # http://localhost:8000/v1
os.environ["CLAIM_LLM_MODEL"]         = MODEL
os.environ["CLAIM_LLM_PROVIDER"]      = "vllm"
os.environ["CLAIM_LLM_SCHEMA_PROFILE"]= SCHEMA_PROFILE   # min_epi -> ~40% fewer output tokens
os.environ["CLAIM_CACHE_PATH"]        = f"{OUT_DIR}/claim_cache.sqlite"  # resume-safe on Drive
# NOTE: do NOT set CLAIM_LLM_EXTRA_BODY reasoning_effort here — that knob is
# LM Studio-specific; vLLM+gemma has no thinking mode and would 400 on it.
print("gemynd routed to", base, "| model", MODEL, "| profile", SCHEMA_PROFILE)


## 7. Dry run — validate the full path on ~5 real docs first

Confirms parse → extract (via vLLM/e4b) → typed claims → bundle, end to end,
before committing to the whole corpus. Bundles land in `{OUT_DIR}_smoke`.


In [ ]:
import sys, glob, json, time
sys.path.insert(0, REPO)
from pathlib import Path
from gemynd.ingest.pipeline import run_e2e, run_e2e_jsonl

sample_json  = sorted(glob.glob(f"{CORPUS_DIR}/**/*.json",  recursive=True))[:5]
sample_jsonl = sorted(glob.glob(f"{CORPUS_DIR}/**/*.jsonl", recursive=True))[:2]
t = time.time()
if sample_jsonl:
    s = run_e2e_jsonl(sample_jsonl, f"{OUT_DIR}_smoke", backend="memory",
                      workers=2, resources_dir=Path(DOMAIN_DIR))
else:
    s = run_e2e(sample_json, f"{OUT_DIR}_smoke", backend="memory",
                workers=2, resources_dir=Path(DOMAIN_DIR))
print(f"dry run {time.time()-t:.1f}s:", {k: s[k] for k in ("documents_processed", "already_ingested") if k in s})
# peek at claims in one bundle
b = sorted(glob.glob(f"{OUT_DIR}_smoke/*.semantic.json"))
if b:
    d = json.load(open(b[0], encoding="utf-8"))
    print("claims in first bundle:", len(d.get("claims", [])))
    for c in d.get("claims", [])[:5]:
        print("  -", c.get("claim_type"), "|", (c.get("normalized_sentence") or c.get("source_sentence") or "")[:70])


## 8. Full ingest — the whole corpus

Uses the CLI so extraction runs across `WORKERS` processes (each feeds vLLM's
batch). `--inputs` on a directory auto-finds and stages both loose `.json` and
one-per-title `.jsonl` bundles. **Resume-safe:** re-running skips docs already in
the checkpoint at `{OUT_DIR}/.ingest_checkpoint.jsonl`.

This is long-running. On Colab, keep the tab active; if the runtime resets, just
re-run this cell — it resumes from the checkpoint on Drive.


In [ ]:
!cd {REPO} && python -m gemynd.cli run-e2e \
    --inputs {CORPUS_DIR} \
    --out-dir {OUT_DIR} \
    --domain-dir {DOMAIN_DIR} \
    --backend memory \
    --workers {WORKERS}


## 9. Progress / throughput — run any time (e.g. in a second session)

In [ ]:
import glob, os, json
done = len(glob.glob(f"{OUT_DIR}/*.semantic.json"))
ckpt = f"{OUT_DIR}/.ingest_checkpoint.jsonl"
n_ckpt = sum(1 for _ in open(ckpt, encoding="utf-8")) if os.path.exists(ckpt) else 0
# rough corpus size (count input docs)
inputs = glob.glob(f"{CORPUS_DIR}/**/*.json", recursive=True) + glob.glob(f"{CORPUS_DIR}/**/*.jsonl", recursive=True)
print(f"bundles written: {done}   checkpoint rows: {n_ckpt}   input files: {len(inputs)}")


## 10. Throughput estimate

`s/para ≈ 2.4 + out_toks/decode_rate`. On a single RX 6600, e4b `min_epi` ran
~4.7 s/para (~32 tok/s, one request at a time). An A100 with vLLM batching runs
**tens of requests concurrently** at a much higher aggregate token rate, so the
effective per-paragraph cost collapses. Measure your real rate below, then
extrapolate to your page/paragraph count.


In [ ]:
import time, json, urllib.request
para = ("The annual convention of the Georgia Woman Suffrage association opened today; "
        "prominent among the subjects to receive attention are child labor and age limits.")
schema = {"type": "json_schema", "json_schema": {"name": "c", "strict": True, "schema": {"type": "array",
    "items": {"type": "object", "additionalProperties": False, "required": ["source_sentence"],
    "properties": {"source_sentence": {"type": "string"}}}}}}
def one():
    req = urllib.request.Request(f"{base}/chat/completions", data=json.dumps({"model": MODEL,
        "messages": [{"role": "user", "content": para}], "max_tokens": 256, "temperature": 0,
        "response_format": schema}).encode(), headers={"Content-Type": "application/json"})
    t = time.time(); urllib.request.urlopen(req, timeout=120).read(); return time.time() - t
import concurrent.futures as cf
N = 32
t = time.time()
with cf.ThreadPoolExecutor(max_workers=N) as ex:
    lat = list(ex.map(lambda _: one(), range(N)))
wall = time.time() - t
print(f"{N} concurrent calls: {wall:.1f}s wall -> {wall/N*1000:.0f} ms/para effective (batched)")
print("estimate for 1M paragraphs at this rate: ~", round(1_000_000 * (wall/N) / 3600, 1), "GPU-hours")


## Next: load bundles into Neo4j

The run produced `*.semantic.json` / `*.structure.json` bundles in your Drive
`OUT_DIR` — the extracted graph, not yet loaded into Neo4j. To load them into a
Neo4j graph (e.g. back on the Pi / your Enterprise instance), point `run-e2e` at
a reachable Neo4j with `--backend neo4j --neo4j-uri … --neo4j-database …`, or run
a dedicated bundle-loader against the bundles. Keep extraction (GPU) and graph
loading (Neo4j host) as separate stages — that's why this notebook writes bundles
rather than requiring Neo4j reachable from Colab.
